# masarch 0.1.0 导入与快速使用

这个 notebook 解决两个最容易混淆的问题：

- **安装名**是 `masarch`
- **导入名**是 `agentorch`

也就是说，如果你要安装 `0.1.0`，命令是：

```bash
pip install masarch==0.1.0
```

但在 Python / Jupyter 里导入时，应该写：

```python
import agentorch
```

说明：

- 这个 notebook 先给你一个**离线可跑**的最小示例，不依赖真实模型 key。
- 最后再给一个**可选的真实模型示例**，只有当前内核已经有 `OPENAI_API_KEY` 和模型名时才运行。
- 在 Jupyter 里请优先使用 `await agent.run(...)`，不要默认用 `run_sync()`。

In [ ]:
# 如果你是在全新 notebook 内核里第一次使用，可以先运行这一格。
# 已经装过就不用重复执行。

# %pip install -U pip
# %pip install masarch==0.1.0

In [ ]:
from __future__ import annotations

import importlib.metadata as metadata
import os
from pathlib import Path

import agentorch

print("导入成功：", agentorch.__name__)
print("导入文件：", agentorch.__file__)

try:
    print("已安装的 masarch 版本：", metadata.version("masarch"))
except metadata.PackageNotFoundError:
    print("当前环境没有通过 pip 安装 masarch，可能是直接从源码目录导入的。")

print("当前工作目录：", Path.cwd())
print("结论：安装名是 masarch，导入名是 agentorch")

## 1. 最小导入后可运行示例

这一段不用真实 API key，目的是确认：

- `import agentorch` 没问题
- `create_agent(...)` 能正常创建
- `await agent.run(...)` 在 notebook 里能正常调用

In [ ]:
from agentorch.core import Message, ModelRequest, ModelResponse, UsageInfo
from agentorch.models.base import BaseModelAdapter


class DummyModel(BaseModelAdapter):
    def __init__(self, *, name: str = "dummy-model", reply: str = "hello from masarch") -> None:
        self.config = {"provider": "dummy", "api_key": "sk-dummy", "model": name}
        self.reply = reply
        self.closed = False

    async def generate(self, request: ModelRequest) -> ModelResponse:
        return ModelResponse(
            message=Message(role="assistant", content=self.reply),
            content=self.reply,
            finish_reason="stop",
            usage=UsageInfo(total_tokens=5),
        )

    async def aclose(self) -> None:
        self.closed = True

In [ ]:
agent = agentorch.create_agent(
    model=DummyModel(reply="minimal agent ok"),
    system_prompt="You are concise.",
    reasoning="react",
)

result = await agent.run(
    "请回复一句最短确认语。",
    thread_id="nb-import-quickstart-001",
)

print("输出：", result.output_text)
print("tokens：", result.usage.total_tokens)
print("blueprint kind：", agent.export_blueprint()["kind"])

await agent.aclose()

## 2. 工具调用示例

这个示例说明导入后不仅能创建 agent，也能挂载工具。

In [ ]:
from pydantic import BaseModel
from agentorch import ToolRegistry, tool


class AddInput(BaseModel):
    a: int
    b: int


@tool(description="Add two integers.")
async def add_numbers(input: AddInput):
    return {"sum": input.a + input.b}


tool_agent = agentorch.create_agent(
    model=DummyModel(reply="tool agent ok"),
    tools=ToolRegistry.from_tools(add_numbers),
    reasoning="react",
)

tool_result = await tool_agent.run(
    "Use add_numbers to compute 12 + 30.",
    thread_id="nb-import-quickstart-002",
)

print("模型输出：", tool_result.output_text)
print("tool_results 数量：", len(tool_result.tool_results))

await tool_agent.aclose()

## 3. 多智能体示例

如果你已经能成功导入并跑通前两格，这一格可以继续确认 `create_multi_agent(...)` 的基本用法。

In [ ]:
planner = agentorch.create_agent(
    model=DummyModel(name="planner-model", reply="planner ready"),
    reasoning="plan_execute",
    name="planner",
)

reviewer = agentorch.create_agent(
    model=DummyModel(name="reviewer-model", reply="reviewer ready"),
    reasoning="react",
    name="reviewer",
)

team = agentorch.create_multi_agent(
    model=DummyModel(name="supervisor-model", reply="supervisor ready"),
    agents=[
        {"agent": planner, "name": "planner", "role": "planner"},
        {"agent": reviewer, "name": "reviewer", "role": "reviewer"},
    ],
    system_prompt="Coordinate specialists and return one final answer.",
    name="demo-team",
)

team_result = await team.run(
    "Draft and review a migration plan.",
    thread_id="nb-import-quickstart-003",
)

print("team 输出：", team_result.output_text)
print("team kind：", team.export_blueprint()["kind"])

await team.aclose()

## 4. 可选：真实模型示例

如果你已经在当前 notebook 内核里设置了下面这些环境变量：

- `OPENAI_API_KEY`
- `OPENAI_CHAT_MODEL` 或 `OPENAI_MODEL` 或 `AGENTORCH_MODEL`
- 如果不是官方接口，再补 `OPENAI_BASE_URL`

就可以运行这一格。否则它会自动跳过。

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
base_url = os.getenv("OPENAI_BASE_URL")
model_name = os.getenv("OPENAI_CHAT_MODEL") or os.getenv("OPENAI_MODEL") or os.getenv("AGENTORCH_MODEL")

if api_key and model_name:
    live_agent = agentorch.create_agent(
        model=agentorch.OpenAIModel(
            api_key=api_key,
            base_url=base_url,
            model=model_name,
        ),
        system_prompt="You are concise.",
        reasoning="react",
    )

    live_result = await live_agent.run(
        "用一句中文介绍你自己。",
        thread_id="nb-import-quickstart-004",
    )

    print("真实模型输出：", live_result.output_text)
    print("真实模型 tokens：", live_result.usage.total_tokens)
    await live_agent.aclose()
else:
    print("跳过真实模型示例：当前内核没有同时提供 OPENAI_API_KEY 和模型名。")

## 5. 结论

如果你只记一件事，就记这个：

```python
# 安装
pip install masarch==0.1.0

# 导入
import agentorch
```

也就是：**装 `masarch`，导 `agentorch`**。